In [ ]:
# pip install pandas requests
import io
import requests
import pandas as pd

In [39]:
START = "1950-01-01"   # earlier than any series starts, so you get each source's full history

def fred(series_id):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text), index_col=0, parse_dates=True, na_values=".")
    return df.iloc[:, 0].loc[START:]

def boc(name):
    url = f"https://www.bankofcanada.ca/valet/observations/{name}/json?start_date={START}"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    obs = r.json()["observations"]
    return pd.Series(
        {pd.Timestamp(o["d"]): float(o[name]["v"]) for o in obs if o.get(name, {}).get("v")}
    )

In [40]:
us_cpi = fred("CPIAUCSL")                        # US CPI index (SA)
ca_cpi = boc("V41690973").resample("MS").last()  # Canada CPI all-items index

series = {
    ("US", "yield_10y"):          fred("DGS10"),
    ("US", "yield_2y"):           fred("DGS2"),
    ("US", "policy_rate"):        fred("DFF"),
    ("US", "inflation_yoy"):      us_cpi.pct_change(12) * 100,
    ("US", "unemployment_rate"):  fred("UNRATE"),
    ("CA", "yield_10y"):          boc("BD.CDN.10YR.DQ.YLD"),
    ("CA", "yield_2y"):           boc("BD.CDN.2YR.DQ.YLD"),
    ("CA", "policy_rate"):        boc("V39079"),           # BoC overnight target
    ("CA", "inflation_yoy"):      ca_cpi.pct_change(12) * 100,
    ("CA", "unemployment_rate"):  fred("LRUNTTTTCAM156S"), # OECD, monthly
}

# 10y - 2y spread (percentage points), computed on dates where both yields exist
for c in ["US", "CA"]:
    series[(c, "spread_10y_2y")] = (series[(c, "yield_10y")] - series[(c, "yield_2y")]).dropna()

/tmp/ipykernel_2610/567373457.py:8: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ("US", "inflation_yoy"):      us_cpi.pct_change(12) * 100,


In [41]:
frames = []
for (country, metric), s in series.items():
    s = s.dropna().loc[START:]
    frames.append(pd.DataFrame({
        "date": s.index.strftime("%Y-%m-%d"),
        "country": country,
        "metric": metric,
        "metric_value": s.values.round(4),
    }))

raw = (pd.concat(frames, ignore_index=True)
         .sort_values(["date", "country", "metric"])
         .reset_index(drop=True))

raw.to_csv("data/macro_raw_long.csv", index=False)   # un-lagged snapshot, so you never need to re-download

# Stacked coverage check: first date, last date, row count per series
print(raw.groupby(["country", "metric"])["date"].agg(["min", "max", "count"]))

                                  min         max  count
country metric                                          
CA      inflation_yoy      1996-01-01  2026-09-01    369
        policy_rate        2009-04-21  2026-11-12   4569
        spread_10y_2y      2001-01-02  2026-11-12   6466
        unemployment_rate  1955-01-01  2026-08-01    860
        yield_10y          2001-01-02  2026-11-12   6467
        yield_2y           2001-01-02  2026-11-12   6466
US      inflation_yoy      1951-01-01  2026-10-01    910
        policy_rate        1954-07-01  2026-11-12  26433
        spread_10y_2y      1976-06-01  2026-11-12  12609
        unemployment_rate  1950-01-01  2026-10-01    921
        yield_10y          1962-01-02  2026-11-12  16201
        yield_2y           1976-06-01  2026-11-12  12609


In [42]:
daily_lag   = 1                                                # all metrics: at least 1 day
release_lag = {"inflation_yoy": 21, "unemployment_rate": 8}    # extra days after reference month ends

long_df = raw.copy()
d = pd.to_datetime(long_df["date"])
long_df["ref_period"] = long_df["date"]          # the period the value describes

# First data point per country and metric (the series' own first observation date, before any lag)
long_df["history_start_date"] = long_df.groupby(["country", "metric"])["ref_period"].transform("min")

is_monthly = long_df["metric"].isin(release_lag)
new_date = d + pd.Timedelta(days=daily_lag)      # daily series: +1 day

month_start_next = d.dt.to_period("M").dt.to_timestamp() + pd.DateOffset(months=1)
extra = long_df["metric"].map(release_lag).fillna(0).astype(int)
new_date = new_date.where(
    ~is_monthly,
    month_start_next + pd.to_timedelta(extra + daily_lag, unit="D"),   # monthly: next month start + release lag + 1
)

long_df["date"] = new_date

In [43]:
MONTHLY = set(release_lag)

def add_z(g):
    g = g.sort_values("date").drop_duplicates("date", keep="last").set_index("date")
    v = g["metric_value"]
    monthly = g["metric"].iloc[0] in MONTHLY

    # 5-year rolling window (calendar-based, past data only, includes current obs)
    r = v.rolling("1826D", min_periods=48 if monthly else 1000)
    g["z_5y"] = (v - r.mean()) / r.std().replace(0, float("nan"))

    # Full history to date (expanding window, no look-ahead)
    e = v.expanding(min_periods=24 if monthly else 250)
    g["z_full"] = (v - e.mean()) / e.std().replace(0, float("nan"))
    return g.reset_index()

z = pd.concat([add_z(g) for _, g in long_df.groupby(["country", "metric"])],
              ignore_index=True)

z = z.sort_values(["date", "country", "metric"]).reset_index(drop=True)
z["date"] = z["date"].dt.strftime("%Y-%m-%d")
z[["z_5y", "z_full"]] = z[["z_5y", "z_full"]].round(4)

In [44]:
cols = ["date", "country", "metric", "metric_value", "z_5y", "z_full",
        "history_start_date", "ref_period"]
final = z[cols]
final.to_csv("data/macro_long_z.csv", index=False)

                          history_start_date    first_z5 first_zfull  \
country metric                                                         
CA      inflation_yoy             1996-01-01  2000-01-23  1998-01-23   
        policy_rate               2009-04-21  2013-02-19  2010-06-02   
        spread_10y_2y             2001-01-02  2004-12-23  2001-12-29   
        unemployment_rate         1955-01-01  1959-01-10  1957-01-10   
        yield_10y                 2001-01-02  2004-12-23  2001-12-29   
        yield_2y                  2001-01-02  2004-12-23  2001-12-29   
US      inflation_yoy             1951-01-01  1955-01-23  1953-01-23   
        policy_rate               1954-07-01  1957-03-27  1955-03-08   
        spread_10y_2y             1976-06-01  1980-06-06  1977-05-28   
        unemployment_rate         1950-01-01  1954-01-10  1952-01-10   
        yield_10y                 1962-01-02  1966-01-06  1963-01-03   
        yield_2y                  1976-06-01  1980-06-06  1977-0